# Portfolio Risk Monitor - Exploratory Data Analysis

This notebook explores the sample data used for the Portfolio Risk Monitor project.
We examine price distributions, technical indicators, sentiment features, and
the target variable to understand the data before model training.

**Author:** Vishal Joshi, MSc Applied AI, University of Warwick

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.collector import SampleDataGenerator
from src.features.technical import TechnicalFeatureEngineer
from src.features.sentiment import SentimentFeatureEngineer
from src.data.preprocessor import DataPreprocessor

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

%matplotlib inline

## 1. Load Sample Data

We use the sample data generator to create synthetic but realistic market data.
This ensures the notebook is reproducible without API keys.

In [ ]:
generator = SampleDataGenerator(sample_path='../data/sample')
prices_df, headlines_df, sentiment_df = generator.load_or_generate()

print(f'Price data shape: {prices_df.shape}')
print(f'Headlines: {len(headlines_df)}')
print(f'Sentiment scores: {len(sentiment_df)}')
print(f'\nTickers: {prices_df["Ticker"].unique().tolist()}')
print(f'Date range: {prices_df["Date"].min()} to {prices_df["Date"].max()}')

prices_df.head()

## 2. Price Data Overview

Let's examine the basic statistics and structure of the price data.

In [ ]:
# Summary statistics per ticker
for ticker in prices_df['Ticker'].unique():
    ticker_data = prices_df[prices_df['Ticker'] == ticker]
    print(f'\n--- {ticker} ---')
    print(f'  Rows: {len(ticker_data)}')
    print(f'  Close range: ${ticker_data["Close"].min():.2f} - ${ticker_data["Close"].max():.2f}')
    print(f'  Mean daily volume: {ticker_data["Volume"].mean():,.0f}')
    daily_ret = ticker_data['Close'].pct_change().dropna()
    print(f'  Mean daily return: {daily_ret.mean():.4%}')
    print(f'  Daily return std: {daily_ret.std():.4%}')
    print(f'  Annualized vol: {daily_ret.std() * np.sqrt(252):.2%}')

In [ ]:
# Plot closing prices
fig, ax = plt.subplots(figsize=(14, 6))
for ticker in prices_df['Ticker'].unique():
    ticker_data = prices_df[prices_df['Ticker'] == ticker]
    # Normalize to base 100 for comparison
    normalized = ticker_data['Close'] / ticker_data['Close'].iloc[0] * 100
    ax.plot(ticker_data['Date'], normalized, label=ticker, linewidth=1.5)

ax.set_title('Normalized Price Comparison (Base = 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Technical Indicators

Compute and visualize key technical indicators for one ticker (SPY).
This helps us understand the features our models will use.

In [ ]:
tech_engineer = TechnicalFeatureEngineer()
technical_features = tech_engineer.compute(prices_df)

# Focus on SPY for visualization
spy_prices = prices_df[prices_df['Ticker'] == 'SPY'].copy().reset_index(drop=True)
spy_tech = technical_features[technical_features['Ticker'] == 'SPY'].copy().reset_index(drop=True)

print(f'Technical features computed: {len(tech_engineer.get_feature_names())} features')
print(f'Feature names: {tech_engineer.get_feature_names()}')

In [ ]:
# Price chart with Bollinger Bands and Moving Averages
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})

# Panel 1: Price with Bollinger Bands
ax = axes[0]
ax.plot(spy_prices['Date'], spy_prices['Close'], 'b-', linewidth=1.2, label='Close')
ax.plot(spy_tech['Date'], spy_tech['bb_upper'], 'r--', alpha=0.5, linewidth=0.8, label='BB Upper')
ax.plot(spy_tech['Date'], spy_tech['bb_lower'], 'g--', alpha=0.5, linewidth=0.8, label='BB Lower')
ax.fill_between(spy_tech['Date'], spy_tech['bb_upper'], spy_tech['bb_lower'], alpha=0.1, color='gray')
ax.plot(spy_tech['Date'], spy_tech['ma_50'], 'orange', alpha=0.7, linewidth=0.8, label='MA 50')
ax.plot(spy_tech['Date'], spy_tech['ma_200'], 'purple', alpha=0.7, linewidth=0.8, label='MA 200')
ax.set_title('SPY - Price with Bollinger Bands & Moving Averages', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: RSI
ax = axes[1]
ax.plot(spy_tech['Date'], spy_tech['rsi'], 'b-', linewidth=1)
ax.axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
ax.axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
ax.fill_between(spy_tech['Date'], 70, spy_tech['rsi'], where=spy_tech['rsi']>70, alpha=0.3, color='red')
ax.fill_between(spy_tech['Date'], 30, spy_tech['rsi'], where=spy_tech['rsi']<30, alpha=0.3, color='green')
ax.set_ylabel('RSI')
ax.set_ylim(0, 100)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: MACD
ax = axes[2]
ax.plot(spy_tech['Date'], spy_tech['macd'], 'b-', linewidth=1, label='MACD')
ax.plot(spy_tech['Date'], spy_tech['macd_signal'], 'r-', linewidth=1, label='Signal')
colors = ['green' if v >= 0 else 'red' for v in spy_tech['macd_histogram'].fillna(0)]
ax.bar(spy_tech['Date'], spy_tech['macd_histogram'], color=colors, alpha=0.3, width=1)
ax.set_ylabel('MACD')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Sentiment Analysis

Examine the pre-computed sentiment scores from financial headlines.

In [ ]:
print(f'Total headlines: {len(sentiment_df)}')
print(f'\nSentiment label distribution:')
print(sentiment_df['label'].value_counts())
print(f'\nCompound score statistics:')
print(sentiment_df['compound_score'].describe())

# Show example headlines for each sentiment
for label in ['positive', 'negative', 'neutral']:
    examples = sentiment_df[sentiment_df['label'] == label].head(3)
    print(f'\n--- {label.upper()} examples ---')
    for _, row in examples.iterrows():
        print(f'  [{row["compound_score"]:.3f}] {row["headline"]}')

In [ ]:
# Sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of compound scores
ax = axes[0]
ax.hist(sentiment_df['compound_score'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Distribution of Compound Sentiment Scores', fontsize=12, fontweight='bold')
ax.set_xlabel('Compound Score (positive - negative)')
ax.set_ylabel('Count')

# Sentiment label pie chart
ax = axes[1]
label_counts = sentiment_df['label'].value_counts()
colors = ['#4CAF50', '#F44336', '#9E9E9E']
ax.pie(label_counts.values, labels=label_counts.index, colors=colors,
       autopct='%1.1f%%', startangle=90)
ax.set_title('Sentiment Label Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Target Variable Analysis

Create and examine the binary target: did the stock drop >2% within the next 5 trading days?

In [ ]:
preprocessor = DataPreprocessor(
    target_threshold=0.02,
    target_horizon=5,
    train_end='2022-12-31',
    val_end='2023-06-30',
)

prices_with_target = preprocessor.create_target(prices_df)

# Class distribution
print('Target Variable Distribution:')
print(prices_with_target['target'].value_counts())
print(f'\nClass imbalance ratio: {prices_with_target["target"].mean():.2%} positive (drop events)')

In [ ]:
# Target distribution per ticker
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall class balance
ax = axes[0]
target_counts = prices_with_target['target'].value_counts()
bars = ax.bar(['No Drop (0)', 'Drop >2% (1)'], target_counts.values,
              color=['#4CAF50', '#F44336'], alpha=0.8)
for bar, count in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count}', ha='center', fontsize=11)
ax.set_title('Target Variable Distribution', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')

# Per-ticker breakdown
ax = axes[1]
ticker_target = prices_with_target.groupby('Ticker')['target'].mean() * 100
ticker_target.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Percentage of >2% Drop Events by Ticker', fontsize=12, fontweight='bold')
ax.set_ylabel('Drop Events (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 6. Feature Correlation Analysis

Examine correlations between technical features and the target variable.
This helps identify which features might be most predictive.

In [ ]:
# Merge technical features with target
sent_engineer = SentimentFeatureEngineer(use_precomputed=True)
scored = sent_engineer.score_headlines(sentiment_df)
tickers = prices_df['Ticker'].unique().tolist()
sentiment_features = sent_engineer.aggregate_daily(scored, tickers=tickers)

merged = preprocessor.merge_features(prices_with_target, technical_features, sentiment_features)
feature_cols = preprocessor.get_feature_columns(merged)

# Compute correlation matrix
corr_cols = feature_cols + ['target']
corr_data = merged[corr_cols].dropna()
corr_matrix = corr_data.corr()

# Plot heatmap of top correlated features
target_corr = corr_matrix['target'].drop('target').abs().sort_values(ascending=False)
top_features = target_corr.head(12).index.tolist()

print('Top features correlated with target:')
for feat in top_features:
    print(f'  {feat}: {corr_matrix.loc[feat, "target"]:.4f}')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
subset_cols = top_features + ['target']
subset_corr = corr_data[subset_cols].corr()

mask = np.triu(np.ones_like(subset_corr, dtype=bool))
sns.heatmap(subset_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap (Top Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Time-Based Split Visualization

Visualize how the data is split into train/validation/test sets.
This is critical to avoid data leakage in financial time series.

In [ ]:
train_df, val_df, test_df = preprocessor.split_data(merged)

print(f'Train: {len(train_df)} samples ({train_df["Date"].min().date()} to {train_df["Date"].max().date()})')
print(f'Val:   {len(val_df)} samples ({val_df["Date"].min().date()} to {val_df["Date"].max().date()})')
print(f'Test:  {len(test_df)} samples ({test_df["Date"].min().date()} to {test_df["Date"].max().date()})')

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
spy_data = merged[merged['Ticker'] == 'SPY']

spy_train = spy_data[spy_data['Date'] <= '2022-12-31']
spy_val = spy_data[(spy_data['Date'] > '2022-12-31') & (spy_data['Date'] <= '2023-06-30')]
spy_test = spy_data[spy_data['Date'] > '2023-06-30']

ax.plot(spy_train['Date'], spy_train['Close'], 'b-', linewidth=1.2, label='Train')
ax.plot(spy_val['Date'], spy_val['Close'], 'orange', linewidth=1.2, label='Validation')
ax.plot(spy_test['Date'], spy_test['Close'], 'g-', linewidth=1.2, label='Test')

ax.axvline(x=pd.Timestamp('2022-12-31'), color='red', linestyle='--', alpha=0.5)
ax.axvline(x=pd.Timestamp('2023-06-30'), color='red', linestyle='--', alpha=0.5)
ax.set_title('SPY Price with Time-Based Train/Val/Test Split', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Key Observations

**Data characteristics:**
- The sample data covers 2021-2023 with realistic price movements generated via geometric Brownian motion
- We have ~750 trading days per ticker across 5 tickers
- The target variable shows moderate class imbalance, which is typical for drawdown prediction

**Feature insights:**
- Volatility and momentum features (RSI, MACD) show meaningful correlation with future drawdowns
- Volume ratios tend to spike before significant price movements
- Sentiment features provide complementary information to purely technical signals

**Modelling considerations:**
- Time-based splitting is essential to prevent look-ahead bias
- Class imbalance will need to be addressed (class weights, oversampling, or threshold tuning)
- The LSTM model's sequence-based approach may capture regime changes that flat features miss

These observations inform our model design choices in the training pipeline.